1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import joblib


2. Upload & Load Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

# Load Excel file
data = pd.read_excel("hydro_power.xlsx")


3. Explore Dataset

In [ ]:
print(data.head())
print(data.info())
print(data.describe())


4. Data Cleaning

In [ ]:
# Drop missing values
data = data.dropna()


5. Feature Engineering & Selection

In [ ]:
X = data[["Rainfall (mm)", "Temperature (C)", "Humidity (%)",
          "Evaporation Loss (mm)", "Water Level (m)",
          "Inflow (cumecs)", "Outflow (cumecs)", "Reservoir Storage (%)"]]

y = data["Power Generation (MW)"]


6. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


7. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


8. Model Training

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


9. Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

print("R² Score:", r2_score(y_test, y_pred))
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))


10. Predict New Data

In [ ]:
sample = [[10, 25, 60, 1.0, 505, 300, 280, 50]]  # Example input
sample_scaled = scaler.transform(sample)
print("Predicted Power:", model.predict(sample_scaled))


11. Save & Load Model

In [ ]:
joblib.dump(model, "hydro_model.pkl")
joblib.dump(scaler, "scaler.pkl")

loaded_model = joblib.load("hydro_model.pkl")
loaded_scaler = joblib.load("scaler.pkl")


12. Deploy with Streamlit

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

model = joblib.load("hydro_model.pkl")
scaler = joblib.load("scaler.pkl")

st.title("Smart Hydro Power Forecast")

rain = st.number_input("Rainfall (mm)", 0.0, 500.0, 10.0)
temp = st.number_input("Temperature (C)", 0.0, 50.0, 25.0)
hum = st.number_input("Humidity (%)", 0.0, 100.0, 60.0)
evap = st.number_input("Evaporation Loss (mm)", 0.0, 10.0, 1.0)
water = st.number_input("Water Level (m)", 400.0, 600.0, 505.0)
inflow = st.number_input("Inflow (cumecs)", 0.0, 2000.0, 500.0)
outflow = st.number_input("Outflow (cumecs)", 0.0, 2000.0, 400.0)
storage = st.number_input("Reservoir Storage (%)", 0.0, 100.0, 50.0)

if st.button("Predict Power Generation"):
    input_data = pd.DataFrame([[rain, temp, hum, evap, water, inflow, outflow, storage]],
        columns=["Rainfall (mm)", "Temperature (C)", "Humidity (%)",
                 "Evaporation Loss (mm)", "Water Level (m)",
                 "Inflow (cumecs)", "Outflow (cumecs)", "Reservoir Storage (%)"])
    input_scaled = scaler.transform(input_data)
    prediction = model.predict(input_scaled)
    st.write("Predicted Power Generation (MW):", round(prediction[0], 2))


13. Run Streamlit App in Colab

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501


/bin/bash: line 1: streamlit: command not found
⠙⠹⠸⠼⠴Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 